| [0.0 Data Collection](00_data_collection.ipynb) | [1.0 Preprocessing](01_data_preprocessing.ipynb) | [2.0 Spatial Autocorrelation](02_spatial_autocorrelation.ipynb) | [3.0 MGWR](03_mgwr_analysis.ipynb) | 4.0 ML Classification |

# 4.0 | Machine Learning Classification — High-Risk MSOA Identification

This notebook develops a **Random Forest classifier** to predict high-risk MSOAs based on  
socio-economic and infrastructure characteristics. The analysis addresses:

1. **Label definition** — top 20% accident rate MSOAs as "high risk"
2. **Class imbalance** — handled via SMOTE oversampling
3. **Spatial awareness** — spatial block cross-validation to prevent data leakage
4. **Evaluation** — precision-recall curve, confusion matrix, feature importance

**Why Random Forest?** It handles non-linear relationships, requires minimal feature engineering,  
and provides interpretable feature importance rankings.

In [ ]:
import sys
sys.path.insert(0, '..')

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import MSOA_ANALYSIS_GPKG, FIGURES_DIR, FIGURE_DPI

plt.rcParams.update({
    'font.family': 'serif',
    'figure.dpi': 150,
    'axes.grid': True,
    'grid.alpha': 0.3
})

In [ ]:
gdf = gpd.read_file(MSOA_ANALYSIS_GPKG)
gdf = gdf[gdf['accident_rate_per_10k'] > 0].copy()

print(f"Loaded {len(gdf):,} MSOAs")
print(f"\nAccident rate percentiles:")
for pct in [25, 50, 75, 80, 90, 95]:
    val = gdf['accident_rate_per_10k'].quantile(pct / 100)
    print(f"  P{pct}: {val:.2f}")

## 4.1 | Define High-Risk Labels

MSOAs in the **top 20th percentile** of accident rates are classified as "high risk" (label = 1).  
This threshold is consistent with commonly used definitions in road safety literature  
and produces a meaningful class imbalance to test the classifier.

In [ ]:
from src.analysis.ml_classification import create_labels

gdf = create_labels(gdf)

print("Class Distribution")
print("═" * 40)
class_counts = gdf['high_risk'].value_counts()
for label, count in class_counts.items():
    pct = count / len(gdf) * 100
    name = 'High Risk' if label == 1 else 'Low Risk'
    print(f"  {name:12s} (label={label}):  {count:5,d}  ({pct:.1f}%)")
print(f"\n  Imbalance ratio: 1:{class_counts[0] / class_counts[1]:.1f}")

### 4.1 | Figure 7 | Class distribution by accident rate

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for label, color, name in [(0, '#3498db', 'Low Risk'), (1, '#e74c3c', 'High Risk')]:
    subset = gdf[gdf['high_risk'] == label]
    subset['accident_rate_per_10k'].hist(bins=50, ax=ax, alpha=0.6, color=color, label=name)

threshold = gdf['accident_rate_per_10k'].quantile(0.8)
ax.axvline(threshold, color='black', linestyle='--', linewidth=2,
           label=f'P80 threshold = {threshold:.1f}')

ax.set_xlabel('Accident Rate per 10,000', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Distribution of Accident Rates by Risk Class', fontweight='bold', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### Feature overview

Summary statistics of features by class. Differences between high-risk and low-risk groups  
indicate which features may be predictive.

In [ ]:
from src.analysis.ml_classification import FEATURE_COLS

available_features = [f for f in FEATURE_COLS if f in gdf.columns]
print(f"Features used ({len(available_features)}): {available_features}\n")

print("Mean values by class:")
print("─" * 65)
print(f"{'Feature':>25s}  {'Low Risk':>12s}  {'High Risk':>12s}  {'Diff %':>10s}")
print("─" * 65)
for feat in available_features:
    low_mean = gdf[gdf['high_risk'] == 0][feat].mean()
    high_mean = gdf[gdf['high_risk'] == 1][feat].mean()
    diff_pct = (high_mean - low_mean) / low_mean * 100 if low_mean != 0 else 0
    print(f"  {feat:>25s}  {low_mean:>12.3f}  {high_mean:>12.3f}  {diff_pct:>+9.1f}%")

## 4.2 | Train Random Forest with Spatial CV + SMOTE

To avoid spatial data leakage, a **spatial block cross-validation** strategy is used:  
MSOAs are grouped into spatial blocks, and entire blocks are held out during each fold.  
This is more conservative than random CV, as it prevents the model from learning  
from geographically proximate training samples.

**SMOTE** (Synthetic Minority Over-sampling Technique) is applied within each fold  
to balance the class distribution and prevent the classifier from defaulting to the majority class.

In [ ]:
from src.analysis.ml_classification import train_random_forest

results = train_random_forest(gdf, spatial_cv=True, use_smote=True)

print("Random Forest Results (Spatial Block CV + SMOTE)")
print("═" * 50)
print(f"  Mean CV F1-score:      {results['mean_f1']:.3f}")
print(f"  Average Precision:     {results['avg_precision']:.3f}")
print(f"")
print(f"  Per-fold F1 scores:")
for i, score in enumerate(results['fold_f1_scores']):
    print(f"    Fold {i+1}: {score:.3f}")
print("═" * 50)

### 4.2 | Figure 8 | Precision-Recall curve

For imbalanced classification problems, the PR curve is more informative than the ROC curve.  
The **Average Precision (AP)** summarises the curve as a single number between 0 and 1.

In [ ]:
from src.visualization.statistical_plots import plot_pr_curve
plot_pr_curve(results['precision'], results['recall'], results['avg_precision'])

# Inline display
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(results['recall'], results['precision'], linewidth=2, color='#e74c3c')
ax.fill_between(results['recall'], results['precision'], alpha=0.1, color='#e74c3c')

# Baseline
baseline = sum(gdf['high_risk'] == 1) / len(gdf)
ax.axhline(baseline, color='grey', linestyle='--', linewidth=1, label=f'Baseline = {baseline:.2f}')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title(f"Precision-Recall Curve (AP = {results['avg_precision']:.3f})",
             fontweight='bold', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

### 4.2 | Figure 9 | Confusion matrix

The confusion matrix shows true vs predicted classifications for the held-out test set.

In [ ]:
from src.visualization.statistical_plots import plot_confusion_matrix
plot_confusion_matrix(results['confusion_matrix'])

# Inline display
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(results['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low Risk', 'High Risk'],
            yticklabels=['Low Risk', 'High Risk'], ax=ax,
            annot_kws={'fontsize': 14})
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

### Classification report

Detailed per-class precision, recall, and F1-score.

In [ ]:
from sklearn.metrics import classification_report

print("Classification Report")
print("═" * 55)
print(classification_report(
    results['y_true'], results['y_pred'],
    target_names=['Low Risk', 'High Risk'],
    digits=3
))

### 4.2 | Figure 10 | Feature importance

Mean Decrease in Impurity (MDI) feature importances from the Random Forest model.  
Higher values indicate greater predictive power for distinguishing high-risk from low-risk MSOAs.

In [ ]:
from src.visualization.statistical_plots import plot_feature_importance
plot_feature_importance(results['feature_importance'])

# Display table and inline plot
fi = results['feature_importance'].sort_values('importance', ascending=False)
print("Feature Importance Ranking")
print("═" * 45)
for rank, (_, row) in enumerate(fi.iterrows(), 1):
    bar = '█' * int(row['importance'] * 50)
    print(f"  {rank}. {row['feature']:25s} {row['importance']:.4f}  {bar}")
print("═" * 45)

fi

In [ ]:
# Horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 6))
fi_sorted = fi.sort_values('importance', ascending=True)
ax.barh(fi_sorted['feature'].str.replace('_', ' ').str.title(),
        fi_sorted['importance'], color='#3498db', edgecolor='white')
for i, v in enumerate(fi_sorted['importance']):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontweight='bold')
ax.set_xlabel('Mean Decrease in Impurity', fontsize=12)
ax.set_title('Random Forest Feature Importance', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

## 4.3 | Spatial Prediction Map

### 4.3 | Figure 11 | Actual vs predicted high-risk MSOAs

Comparing actual high-risk MSOAs (from data) with model predictions.  
Spatial consistency between the two maps indicates the model captures genuine spatial patterns.

In [ ]:
# Attach predictions to GeoDataFrame
mask = gdf[available_features + ['high_risk']].dropna().index
gdf_pred = gdf.loc[mask].copy()
gdf_pred['predicted_risk'] = results['y_pred']
gdf_pred['risk_probability'] = results['y_pred_proba']

print(f"Predictions attached to {len(gdf_pred):,} MSOAs")
print(f"  Actual high-risk:    {sum(gdf_pred['high_risk'] == 1):,}")
print(f"  Predicted high-risk: {sum(gdf_pred['predicted_risk'] == 1):,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 10))

# Actual
gdf_pred.plot(column='high_risk', cmap='RdYlGn_r', legend=True, ax=axes[0],
              edgecolor='face', linewidth=0.1,
              legend_kwds={'label': 'Risk Class', 'shrink': 0.6})
axes[0].set_title('Actual High-Risk MSOAs (Top 20%)', fontweight='bold', fontsize=12)
axes[0].axis('off')

# Predicted probability
gdf_pred.plot(column='risk_probability', cmap='RdYlGn_r', legend=True, ax=axes[1],
              edgecolor='face', linewidth=0.1,
              legend_kwds={'label': 'Predicted Probability', 'shrink': 0.6})
axes[1].set_title('Predicted Risk Probability', fontweight='bold', fontsize=12)
axes[1].axis('off')

plt.suptitle('Random Forest: Actual vs Predicted Risk', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

## 4.4 | Summary

**Key findings from ML classification:**

1. A Random Forest classifier with spatial block CV and SMOTE can identify high-risk  
   MSOAs from socio-economic and infrastructure features.

2. The model's performance (as measured by Average Precision and F1-score) demonstrates  
   that area-level characteristics carry meaningful predictive signal for accident risk.

3. Feature importance analysis reveals which factors are most predictive of high-risk  
   classification, providing actionable insights for road safety policy.

4. Spatial CV ensures model evaluation reflects real-world prediction scenarios where  
   the model must generalise to new geographical areas.

---

**Next steps**: Combine findings from spatial autocorrelation (Notebook 02), MGWR (Notebook 03),  
and ML classification (this notebook) to produce the final analytical report.

---
| [0.0 Data Collection](00_data_collection.ipynb) | [1.0 Preprocessing](01_data_preprocessing.ipynb) | [2.0 Spatial Autocorrelation](02_spatial_autocorrelation.ipynb) | [3.0 MGWR](03_mgwr_analysis.ipynb) | 4.0 ML Classification |